# CUDA-Q × qBraid: Quickstart

[qBraid](https://www.qbraid.com/) is a cloud platform that brokers access to quantum simulators and QPUs from multiple vendors (Rigetti, IonQ, IQM, and more) behind a single API. As of [NVIDIA/cuda-quantum#4328](https://github.com/NVIDIA/cuda-quantum/pull/4328), `qbraid` is a first-class remote REST target in CUDA-Q — you write a `@cudaq.kernel`, set the target to `qbraid`, and the kernel is lowered to OpenQASM 2.0, submitted to the qBraid v2 API, and the measurement counts are returned as a `cudaq.SampleResult`.

There is **no Python dependency on `qbraid-sdk`** — the integration is implemented inside `nvq++` itself.

This notebook is a tour of the integration:
1. Prereqs and credential setup
2. `cudaq.set_target("qbraid", ...)` arguments
3. `cudaq.sample` on the default qBraid QIR simulator
4. `cudaq.sample_async` and persisting futures across processes
5. `cudaq.observe` for expectation values (VQE-style)
6. Switching to a real QPU
7. Error handling reference

## 1. Prereqs

- A CUDA-Q build that includes the qBraid target 
- A qBraid account and API key from <https://account.qbraid.com/>

Export the API key once per shell:

```bash
export QBRAID_API_KEY="qbraid_generated_api_key"
```

or pass it inline via the `api_key` argument (shown below).

In [ ]:
# SKIP_CI: requires CUDA-Q runtime and qBraid API credentials
import os
import cudaq

assert os.environ.get(
    "QBRAID_API_KEY"
), "Set QBRAID_API_KEY in your environment, or pass api_key=... to set_target below."

print("CUDA-Q version:", cudaq.__version__)

## 2. Setting the target

The qBraid target accepts two arguments:

| Argument | Purpose | Default |
|----------|---------|---------|
| `machine` | qBraid device ID (the `deviceQrn`) | `qbraid:qbraid:sim:qir-sv` |
| `api_key` | qBraid API key; overrides `QBRAID_API_KEY` | `$QBRAID_API_KEY` |

The full catalog of device IDs lives at <https://account.qbraid.com/devices>.

Default — submits to qBraid's QIR state-vector simulator:

In [ ]:
cudaq.set_target("qbraid")

Explicit — useful when you want to lock the device in source:

In [ ]:
cudaq.set_target(
    "qbraid",
    machine="qbraid:qbraid:sim:qir-sv",
    # api_key="...",  # only if you'd rather not use the env var
)

> **Note on `emulate`.** qBraid devices are cloud-hosted, so there is no local-emulation path — every call submits to the qBraid service. To run "for free" while developing, pick one of the qBraid *simulator* device IDs (the default is one) rather than a QPU.

## 3. `cudaq.sample`: synchronous submission

A 2-qubit Bell state is the simplest sanity check — measure outcomes should cluster around `00` and `11`.

In [ ]:
@cudaq.kernel
def bell():
    q = cudaq.qvector(2)
    h(q[0])
    x.ctrl(q[0], q[1])
    mz(q)

In [ ]:
counts = cudaq.sample(bell, shots_count=2000)
print(counts)

`shots_count` defaults to 1000 if omitted.

## 4. `cudaq.sample_async`: queue-friendly submission

Real QPU runs sit in a vendor queue — sometimes for minutes, sometimes for hours. `sample_async` returns a future immediately so your Python process is free to do other work (or even exit and come back later).

### 4a. Wait inline

In [ ]:
future = cudaq.sample_async(bell, shots_count=1000)
# ... classical work happens here while qBraid is running the job ...
counts = future.get()
print(counts)

### 4b. Persist the future, read it back from a different process

`str(future)` is a complete handle: device ID, job ID, headers, everything needed to re-fetch results. Write it to disk and you can pick the result up tomorrow.

In [ ]:
future = cudaq.sample_async(bell, shots_count=1000)

with open("qbraid_job.json", "w") as f:
    f.write(str(future))

# ... later, possibly in a different script ...

with open("qbraid_job.json") as f:
    handle = f.read()

rehydrated = cudaq.AsyncSampleResult(handle)
print(rehydrated.get())

## 5. `cudaq.observe`: expectation values

Sampling isn't the only thing the integration supports. `observe` works too — useful for VQE, QAOA, and other variational workflows.

In [ ]:
from cudaq import spin


@cudaq.kernel
def ansatz(theta: float):
    q = cudaq.qvector(2)
    x(q[0])
    ry(theta, q[1])
    x.ctrl(q[1], q[0])


hamiltonian = (
    5.907
    - 2.1433 * spin.x(0) * spin.x(1)
    - 2.1433 * spin.y(0) * spin.y(1)
    + 0.21829 * spin.z(0)
    - 6.125 * spin.z(1)
)

result = cudaq.observe(ansatz, hamiltonian, 0.59, shots_count=4000)
print("<H> =", result.expectation())

`observe_async` exists too and serializes/rehydrates the same way — use `cudaq.AsyncObserveResult(handle, hamiltonian)` to read it back.

## 6. Switching to a real QPU

Every device on qBraid is selected by its `deviceQrn` string. Browse the live catalog at <https://account.qbraid.com/devices> — the IDs follow the pattern `<provider>:<vendor>:<type>:<name>`. A few illustrative ones:

| Device | `machine` value |
|--------|-----------------|
| qBraid QIR state-vector simulator (default) | `qbraid:qbraid:sim:qir-sv` |
| Amazon Braket SV1 (cloud simulator) | `aws:aws:sim:sv1` |
| IonQ, Rigetti, IQM QPUs | see the qBraid device catalog for current IDs |

Submitting the same `bell` kernel to a different device is a one-line swap:

```python
cudaq.set_target("qbraid", machine="<paste-qpu-device-id-here>")
counts = cudaq.sample(bell, shots_count=1000)
```

Real QPUs charge per shot and can have long queues — use `sample_async` and persist the future.

## 7. Error handling

The integration translates qBraid v2 HTTP responses into Python exceptions you can act on:

| HTTP status from qBraid | What `cudaq.sample` raises | Meaning |
|---|---|---|
| `401` / `403` | `RuntimeError: qBraid authentication failed (HTTP 4xx)` | Bad / missing API key. Check `QBRAID_API_KEY` or the `api_key` arg. |
| `404` on `/result` | `RuntimeError: qBraid result not found` | The job was deleted or never produced results. |
| `409` on `/result` | `RuntimeError: qBraid job ... did not produce results` | Job ended in `FAILED` or `CANCELLED`. |
| `5xx`, network blips, JSON parse | retried with exponential backoff (3 attempts) | Transient. |

There's also a *non-error* race the helper absorbs for you: qBraid's status transitions to `COMPLETED` slightly before `/result` becomes queryable. The helper retries with backoff so you never see it.

## Where next

- `cuda_quantum_qpu_benchmarking.ipynb` — a deeper walkthrough that uses this same target to benchmark a noisy QPU against the qBraid QIR simulator for a specific application kernel.
- `qbraid_example.cpp` — the same `bell` kernel in C++, compiled with `nvq++ --target qbraid`.
- Reference: <https://nvidia.github.io/cuda-quantum/latest/using/backends/cloud/qbraid.html>

<div class="alert alert-block alert-info">
<b>Copyright Notice:</b> 
    All rights reserved © [2026] qBraid. This notebook is part of the qBraid-Lab-Demo repository.
The qBraid-Lab-Demo is licensed under the Apache License, Version 2.0.
You may obtain a copy of the License at <https://www.apache.org/licenses/LICENSE-2.0>.
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
</div>